In [2]:
import networkx as nx
import numpy as np

## a mudar:

- alphaL,alphaR, gamma, beta fixo
- escolha de influencers só tem a de maior número de arestas (e particularmente 1 de cada cor)
- ele não salva por t

In [ ]:
class GraphModel:

    def __init__(self, num_nodes, num_words, tau):

        self.num_nodes = num_nodes
        self.num_words = num_words
        self.tau = tau

        # alphaL and alphaR parameters (fixed for now)
        self.alphaL = 1
        self.alphaR = 1

        # gamma and beta parameters (fixed for now)
        self.gamma = 0.2
        self.beta = 0.3

        # word dictionary
        random_vector1 = np.random.rand(num_words)
        self.dictionary = []
        for i in range(num_words):
            self.dictionary.append("blue" if random_vector1[i] > 0.5 else "green")

        # word count by color
        self.num_green_words = self.dictionary.count("green")
        self.num_blue_words = self.dictionary.count("blue")

        # graph initialization
        self.G = nx.Graph()
        random_vector2 = np.random.rand(num_nodes)

        for i in range(num_nodes):
            self.G.add_node(
                i,
                color="green" if random_vector2[i] > 0.5 else "blue"
            )

        # probability vector for each node
        for i in range(num_nodes):

            if self.G.nodes[i]['color'] == "green":
                probs = np.random.dirichlet(
                    self.alphaL * np.ones(self.num_green_words)
                )
            else:
                probs = np.random.dirichlet(
                    self.alphaR * np.ones(self.num_blue_words)
                )

            prob_vector = np.zeros(num_words)
            idx = 0

            for j in range(num_words):
                if self.dictionary[j] == self.G.nodes[i]['color']:
                    prob_vector[j] = probs[idx]
                    idx += 1

            self.G.nodes[i]['prob'] = prob_vector

        # initial edges
        for i in range(num_nodes):
            for j in range(i + 1, num_nodes):
                self.update_edge(i, j)


    def run_cycles(self, num_cycles):
        for _ in range(num_cycles):
            self.step_1() #parameters: selection function S, the influencer's susceptibility beta and message type p_msg 
            self.step_2() #parameters: the users' susceptibility gamma
            self.step_3() #parameters: edge update threshold tau


    def step_1(self):
        self.influencer_nodes = []
        # For instance, the influencer selection will be the two nodes with the highest degree (one of each color)

        max_degrees = [(-1, -1), (-1,-1)] # (degree, node_index) for green and blue
        for i in range(self.num_nodes):
            degree, color = self.G.degree(i), self.G.nodes[i]['color']
            if color == "green" and degree > max_degrees[0][0]:
                max_degrees[0] = (degree, i)
            elif color == "blue" and degree > max_degrees[1][0]:
                max_degrees[1] = (degree, i)

        self.influencer_nodes = [max_degrees[0][1], max_degrees[1][1]]
            
        p_msg = np.ones(self.num_words) / self.num_words    
        for i in self.influencer_nodes:
            self.G.nodes[i]['prob'] = (1 - self.beta) * self.G.nodes[i]['prob'] + self.beta * p_msg

        print(f"Influencers selected: {self.influencer_nodes}") # debugging print

    def step_2(self): 

        for i in range(self.num_nodes):
            if i not in self.influencer_nodes:

                Nu = []  # Nu: influencers connected to user i
                for j in self.influencer_nodes:
                    if self.G.has_edge(i, j):
                        Nu.append(j)

                if len(Nu) > 0:
                    profile_sum = np.zeros(self.num_words)
                    for k in Nu:
                        profile_sum += self.G.nodes[k]['prob'] / len(Nu)
                else:
                    profile_sum = self.G.nodes[i]['prob']

                # The new user profile is a convex combination of the current profile and the average influencer profile
                self.G.nodes[i]['prob'] = (
                    (1 - self.gamma) * self.G.nodes[i]['prob'] + self.gamma * profile_sum
                )


    def step_3(self):

        for i in range(self.num_nodes):
            for j in range(i + 1, self.num_nodes):
                self.update_edge(i, j)

    
    def update_edge(self, node1, node2):

        v1 = self.G.nodes[node1]['prob']
        v2 = self.G.nodes[node2]['prob']

        dot_product = np.dot(v1, v2)

        if dot_product > self.tau:
            self.G.add_edge(node1, node2)
        else:
            if self.G.has_edge(node1, node2):
                self.G.remove_edge(node1, node2)


    def cross_group_conection(self): 
        #a measure of bubble burst by counting the fraction of cross-group edges
        total_cross_group_edges_possible = self.num_blue_words * self.num_green_words
        cross_group_edges = 0

        for node1 in range(self.num_nodes):
            for node2 in range(node1 + 1, self.num_nodes):
                color1 = self.G.nodes[node1]['color']
                color2 = self.G.nodes[node2]['color']

                if color1 != color2 and self.G.has_edge(node1, node2):
                        cross_group_edges += 1

        return cross_group_edges / total_cross_group_edges_possible
    
    def teste(self): # usada para verificar o valor tau na prática
        for i in range(10):
            print(f"Node {i}: Color = {self.G.nodes[i]['color']}, edges = {list(self.G.neighbors(i))}")


In [91]:
np.random.seed(42)
teste_0 = GraphModel(2000, 200, 0.1) #o tau tem que mudar conforme a escala, principalmente do num_words pq afeta o produto escalar

teste_0.teste()

# teste_0.run_cycles(1)
# teste_0
# teste_0.teste()

teste_0.cross_group_conection()

Node 0: Color = green, edges = []
Node 1: Color = blue, edges = []
Node 2: Color = blue, edges = []
Node 3: Color = green, edges = []
Node 4: Color = green, edges = []
Node 5: Color = blue, edges = []
Node 6: Color = blue, edges = []
Node 7: Color = green, edges = []
Node 8: Color = blue, edges = []
Node 9: Color = blue, edges = []


0.0